In [15]:
import pandas as pd

In [16]:
merged_df = pd.read_csv(r"C:\Users\dhrub\Desktop\Dengue_summary_report\merged_output1.csv")

In [17]:
merged_df[['year', 'month']].dtypes


year     int64
month    int64
dtype: object

In [18]:
merged_df[merged_df['year'] == 2019].shape


(2147, 5)

In [20]:
merged_df[
    (merged_df['year'] == 2019) &
    (merged_df['District'] == 'Kaski')
].shape


(0, 5)

In [22]:
merged_df['District'] = merged_df['District'].str.strip().str.title()


In [24]:
merged_df[
    (merged_df['year'] == 2019) &
    (merged_df['District'] == 'Kaski')
].shape


(2147, 5)

In [25]:
merged_df.loc[merged_df['year'] == 2019, 'District'].value_counts()


District
Kaski    2147
Name: count, dtype: int64

In [26]:
merged_df.loc[merged_df['year'] == 2019, 'month'].unique()


array([ 8,  7,  9, 10, 11, 12], dtype=int64)

In [27]:
merged_df[merged_df['year'] == 2019].head()


,Ward Number,District,Municipality,year,month
5912,8.0,Kaski,40504,2019,8
5913,8.0,Kaski,40504,2019,8
5914,9.0,Kaski,40504,2019,8
5915,8.0,Kaski,40504,2019,8
5916,8.0,Kaski,40504,2019,8


In [29]:
merged_df[['year', 'month']].head(20)


,year,month
0,2025,9
1,2025,9
2,2025,9
3,2025,9
4,2025,9
5,2025,9
6,2025,9
7,2025,9
8,2025,7
9,2025,7


In [30]:
# -------------------------------
# 1. Filter Kaski district (2019–2025)
# -------------------------------
kaski_df = merged_df[
    (merged_df['District'] == 'Kaski') &
    (merged_df['year'].between(2019, 2025))
].copy()

# Ensure year & month are integers
kaski_df['year'] = kaski_df['year'].astype(int)
kaski_df['month'] = kaski_df['month'].astype(int)

# -------------------------------
# 2. Monthly aggregation (1 row = 1 case)
# -------------------------------
monthly_cases = (
    kaski_df
    .groupby(['year', 'month'])
    .size()
    .reset_index(name='total_cases')
)

# -------------------------------
# 3. Create complete year–month grid
# -------------------------------
all_years = range(2019, 2026)   # 2019 to 2025
all_months = range(1, 13)       # Jan to Dec

full_index = pd.MultiIndex.from_product(
    [all_years, all_months],
    names=['year', 'month']
)

full_df = pd.DataFrame(index=full_index).reset_index()

# -------------------------------
# 4. Merge & fill missing months with 0
# -------------------------------
monthly_cases_full = (
    full_df
    .merge(monthly_cases, on=['year', 'month'], how='left')
    .fillna({'total_cases': 0})
)

monthly_cases_full['total_cases'] = monthly_cases_full['total_cases'].astype(int)

# -------------------------------
# 5. Sort & add date column
# -------------------------------
monthly_cases_full = monthly_cases_full.sort_values(['year', 'month'])

monthly_cases_full['date'] = pd.to_datetime(
    monthly_cases_full['year'].astype(str) + '-' +
    monthly_cases_full['month'].astype(str).str.zfill(2) + '-01'
)

# -------------------------------
# 6. Export to Excel (optional)
# -------------------------------
monthly_cases_full.to_csv(
    'kaski_monthly_dengue_2019_2025.csv',
    index=False
)

# -------------------
